# 04 — Run Vertex AI no-memory baseline

This runs the same fixed 12-task manifest on Vertex AI with Application Default Credentials (ADC). It uses project `dsi-tifin-harness`, location `global`, and `gemini-3.5-flash-lite`. No API key is used.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'src' / 'state_management').is_dir() else WORKING_DIR.parent
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
RUNS_DIR = EXPERIMENTS_DIR / 'runs'
MANIFEST_PATH = RUNS_DIR / 'tasks.jsonl'
RESULTS_PATH = RUNS_DIR / 'vertex_results.jsonl'
RUNNER_PATH = EXPERIMENTS_DIR / 'run_vertex_no_memory_baseline.py'
if not MANIFEST_PATH.is_file():
    raise RuntimeError('Run 01_build_task_manifest.ipynb first.')
print('Project:', os.environ.get('VERTEX_AI_PROJECT_ID', 'dsi-tifin-harness'))
print('Location:', os.environ.get('VERTEX_AI_LOCATION', 'global'))
print('Model:', os.environ.get('VERTEX_MODEL', 'gemini-3.5-flash-lite'))
print('This sends 12 paid Vertex AI requests and saves actual answers and metrics.')

In [ ]:
environment = os.environ.copy()
environment.setdefault('VERTEX_AI_PROJECT_ID', 'dsi-tifin-harness')
environment.setdefault('VERTEX_AI_LOCATION', 'global')
environment.setdefault('VERTEX_MODEL', 'gemini-3.5-flash-lite')
run = subprocess.run(
    [sys.executable, str(RUNNER_PATH), str(MANIFEST_PATH), str(RESULTS_PATH)],
    cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True, check=False,
)
print(run.stdout)
if run.returncode:
    print(run.stderr)
    raise RuntimeError('Vertex baseline stopped. Completed task records were preserved; re-run after fixing the error.')

In [ ]:
results = [json.loads(line) for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines() if line]
print(f'Completed Vertex baseline records: {len(results)} / 12')
for record in results:
    print(record['task_id'], '|', record['latency_ms'], 'ms |', record['total_tokens'], 'tokens |', record['answer_path'])
if len(results) != 12:
    raise RuntimeError('Baseline incomplete. Re-run the previous cell; completed tasks will be skipped.')